In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "rte"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 3000/3000 [00:00<00:00, 277199.39 examples/s]


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3000
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 2490
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3000
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'No Weapons of Mass Destruction Found in Iraq Yet.',
 'sentence2': 'Weapons of Mass Destruction Found in Iraq.',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Britain said, Friday, that it has barred cleric, Omar Bakri, from returning to the country from Lebanon, where he was released by police after being detained for 24 hours.',
 'Bakri was briefly detained, but was released.')

In [11]:
raw_train_dataset[5]['label']

0

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['entailment', 'not_entailment']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 3000/3000 [00:00<00:00, 29241.71 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 167]),
 'token_type_ids': torch.Size([32, 167]),
 'attention_mask': torch.Size([32, 167])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.8105, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt')
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.6859205776173285}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.6859205776173285}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

10

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/',
    cka_max_iter=150
)

Target Metric for rte: accuracy
Original accuracy score: 0.6859
Recovery threshold: 0.0069

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 78/78 [00:08<00:00,  9.29it/s]


  Similarity Stats:
    Average: 0.877856
    Highest: 0.960551
    Lowest:  0.570555

[2/5] Merging Layers...
  Merged Layers: 6 + 7
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6, 7], [8], [9], [10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.6751

[4/5] Recovery Training...
  Performance drop: 0.0108 (threshold: 0.0069)
  → Recovery training NEEDED
  → Training: 10 epochs, lr=1e-05


Training:   0%|          | 0/780 [00:00<?, ?it/s]/home/abhinavl/work/Layer_Graft/layer-grafting-publication/similar_layer_merge/../train_eval_func.py:223: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|█         | 78/780 [00:17<02:25,  4.82it/s]

✓ Saved checkpoint (best loss: 0.9733)

<----------------- Epoch 1 ----------------->
Loss: 0.16, Training Metrics:
  accuracy: 0.9470
Validation Loss: 0.97, Validation Metrics:
  accuracy: 0.6715
Elapsed Time: 19.3195 sec


...

Training:  30%|███       | 234/780 [01:19<01:47,  5.09it/s]


<----------------- Epoch 3 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9618
Validation Loss: 1.46, Validation Metrics:
  accuracy: 0.6462
Elapsed Time: 18.9246 sec


Training:  40%|████      | 312/780 [01:39<02:28,  3.15it/s]


<----------------- Epoch 4 ----------------->
Loss: 0.07, Training Metrics:
  accuracy: 0.9731
Validation Loss: 1.76, Validation Metrics:
  accuracy: 0.6643
Elapsed Time: 18.6416 sec
>>>>> Early stopping callback <<<<<

Total Training Time: 99.19 sec
Chosen Primary Metric: accuracy
Best Metric Run: 0.6751

  → Loading best checkpoint...


  ✓ Loaded best checkpoint from epoch 2
    Best val loss: 1.3131
    Best val metrics: {'accuracy': 0.6750902527075813}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-rte.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (6, 7)
    Layers after: 11
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.6967509025270758}
  Iteration 2:
    Merged: (7, 8)
    Layers after: 10
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.7148014440433214}
  Iteration 3:
    Merged: (7, 8)
    Layers after: 9
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.6931407942238267}
  Iteration 4:
    Merged: (5, 6)
    Layers after: 8
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.6750902527075813}
  Iteration 5:
    Merged: (1, 2)
    Layers after: 7
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.6931407942238267}
  Iteration 6:
    Merged: (2, 3)
    Layers after: 6
    Composition: [[0], [1, 2], [3, 4], [5, 6, 7], [8, 9, 10